In [1]:
import sys
import json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

DEMO = ROOT / "demo"
sys.path.insert(0, str(ROOT))   

from utils import normalize_ar

print("ROOT:", ROOT)
print("DEMO:", DEMO)

ROOT: /home/joox/Desktop/egyptian-personal-status-rag
DEMO: /home/joox/Desktop/egyptian-personal-status-rag/demo


In [2]:
chunks = json.load(open(ROOT / "data" / "chunks.json", encoding="utf-8"))

print("عدد المواد:", len(chunks))
print("الـ keys:", list(chunks[0].keys()))
print()
print(chunks[0]["id"])
print(chunks[0]["text"])

عدد المواد: 188
الـ keys: ['id', 'text', 'law', 'article', 'page', 'law_name', 'heading', 'n_words']

law25_1920_a001
مادة 1
تجب النفقة للزوجة على زوجها من تاريخ العقد الصحيح إذا سلمت نفسها إليه ولو حكما
حتى لو كانت موسرة أو مختلفة معه فى الدين.
ولا يمنع مرض الزوجة من استحقاقها للنفقة.
وتشمل النفقة الغذاء والكسوة والمسكن ومصاريف العلاج وغير ذلك مما يقضى به الشرع.
ولا يجب النفقة للزوجة إذا ارتدت، او امتنعت مختارة من تسليم نفسها دون حق او
اضطرت الى ذلك بسبب ليس من قبل الزوج، او خرجت دون اذن زوجها.
ولا يعتبر سببا لسقوط نفقة الزوجة خروجها من مسكن الزوجية- دون إذن زوجها - فى
الأحوال التي يباح فيها ذلك بحكم الشرع مما ورد فيه نص او جرى به عرف او قضت به
ضرورة، ولا خروجها للعمل المشروط مشوب بإساءة استعمال الحق، او مناف لمصلحة
الأسرة وطلب منها الزوج الامتناع عنه.
وتعتبر نفقة الزوجة دينا على الزوج من تاريخ امتناعه عن الإنفاق مع وجوبه . ولا تسقط
إلا بالإدلاء او الإبراء .
ولا تسمع دعوى النفقة عن مدة ماضية لأكثر من سنة نهايتها تاريخ رفع الدعوى.
ولا يقبل من الزوج التمسك بالمقاصة بين نفقة الزوجة وبين 

In [3]:
sample = chunks[0]["text"][:200]

print("قبل:")
print(sample)
print()
print("بعد:")
print(normalize_ar(sample))

قبل:
مادة 1
تجب النفقة للزوجة على زوجها من تاريخ العقد الصحيح إذا سلمت نفسها إليه ولو حكما
حتى لو كانت موسرة أو مختلفة معه فى الدين.
ولا يمنع مرض الزوجة من استحقاقها للنفقة.
وتشمل النفقة الغذاء والكسوة وال

بعد:
مادة 1
تجب النفقة للزوجة على زوجها من تاريخ العقد الصحيح اذا سلمت نفسها اليه ولو حكما
حتى لو كانت موسرة او مختلفة معه فى الدين.
ولا يمنع مرض الزوجة من استحقاقها للنفقة.
وتشمل النفقة الغذاء والكسوة وال


In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")
model.max_seq_length = 512

print("الموديل اتحمل على:", model.device)
print("max_seq_length:", model.max_seq_length)
print("الـ dim:", model.get_sentence_embedding_dimension())

/home/joox/Desktop/egyptian-personal-status-rag/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2708.76it/s]


الموديل اتحمل على: cuda:0
max_seq_length: 512
الـ dim: 768


/tmp/ipykernel_2384123/3673007940.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("الـ dim:", model.get_sentence_embedding_dimension())


In [5]:
question = "هل الزوجة ليها نفقة لو عيانة؟"
texts = [
    "ولا يمنع مرض الزوجة من استحقاقها للنفقة.",
    "تنتهي حضانة النساء ببلوغ الصغير سن الخامسة عشرة.",
]

q_vec = model.encode(["query: " + normalize_ar(question)], normalize_embeddings=True)
p_vecs = model.encode(["passage: " + normalize_ar(t) for t in texts], normalize_embeddings=True)

print("shape السؤال:", q_vec.shape)
print("shape المواد:", p_vecs.shape)
print("الـ dtype:", p_vecs.dtype)
print()

scores = p_vecs @ q_vec[0]
for t, s in zip(texts, scores):
    print(f"{s:.3f}  {t}")

shape السؤال: (1, 768)
shape المواد: (2, 768)
الـ dtype: float32

0.821  ولا يمنع مرض الزوجة من استحقاقها للنفقة.
0.772  تنتهي حضانة النساء ببلوغ الصغير سن الخامسة عشرة.


In [6]:
question = "هل الزوجة ليها نفقة لو عيانة؟"
texts = [
    "ولا يمنع مرض الزوجة من استحقاقها للنفقة.",
    "تنتهي حضانة النساء ببلوغ الصغير سن الخامسة عشرة.",
]

q_vec = model.encode([ normalize_ar(question)], normalize_embeddings=True)
p_vecs = model.encode([normalize_ar(t) for t in texts], normalize_embeddings=True)

print("shape السؤال:", q_vec.shape)
print("shape المواد:", p_vecs.shape)
print("الـ dtype:", p_vecs.dtype)
print()

scores = p_vecs @ q_vec[0]
for t, s in zip(texts, scores):
    print(f"{s:.3f}  {t}")

shape السؤال: (1, 768)
shape المواد: (2, 768)
الـ dtype: float32

0.856  ولا يمنع مرض الزوجة من استحقاقها للنفقة.
0.802  تنتهي حضانة النساء ببلوغ الصغير سن الخامسة عشرة.


In [7]:
tokenizer = model.tokenizer

lengths = []
for c in chunks:
    text = "passage: " + normalize_ar(c["text"])
    n_tokens = len(tokenizer(text)["input_ids"])
    lengths.append(n_tokens)

too_long = [(c["id"], n) for c, n in zip(chunks, lengths) if n > model.max_seq_length]

print("أطول مادة:", max(lengths), "token")
print("متوسط الطول:", sum(lengths) // len(lengths), "token")
print("عدد المواد اللي هتتقص:", len(too_long), "من", len(chunks))
print()
for cid, n in sorted(too_long, key=lambda x: -x[1]):
    print(f"  {cid}: {n} token")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (715 > 512). Running this sequence through the model will result in indexing errors


أطول مادة: 836 token
متوسط الطول: 109 token
عدد المواد اللي هتتقص: 3 من 188

  decree1090_2000_a002: 836 token
  law1_2000_a009: 715 token
  law1_2000_a015: 592 token


In [8]:
passages = ["passage: " + normalize_ar(c["text"]) for c in chunks]

embeddings = model.encode(
    passages,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("shape:", embeddings.shape)
print("dtype:", embeddings.dtype)
print("طول أول vector:", (embeddings[0] ** 2).sum() ** 0.5)

Batches: 100%|██████████| 6/6 [00:01<00:00,  4.21it/s]

shape: (188, 768)
dtype: float32
طول أول vector: 1.0


In [ ]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print( dim)
print( index.ntotal)

الـ dim: 768
عدد الـ vectors في الـ index: 188


In [10]:
question = "هل يمكن حبس من يمتنع عن دفع النفقة؟"

q_vec = model.encode(
    ["query: " + normalize_ar(question)],
    normalize_embeddings=True,
)

scores, positions = index.search(q_vec, 5)

print("السؤال:", question)
print()
for rank, (pos, score) in enumerate(zip(positions[0], scores[0]), start=1):
    c = chunks[pos]
    preview = c["text"].replace("\n", " ")[:80]
    print(f"{rank}. {c['id']}  ({score:.3f})")
    print(f"   {preview}...")

السؤال: هل يمكن حبس من يمتنع عن دفع النفقة؟

1. law1_2000_a076bis  (0.851)
   تنفيذ حكم النفقة مادة 76 مكرر إذا امتنع المحكوم عليه عن تنفيذ الحكم النهائي الصا...
2. law25_1920_a004  (0.846)
   القسم الثاني فى العجز عن النفقة مادة 4 إذا امتنع الزوج عن الإنفاق على زوجته، فان...
3. law1_2000_a079  (0.842)
   عقوبة صرف نفقة غير مستحقة من بنك ناصر مادة 79 مع عدم الإخلال باى عقوبة اشد ينص ع...
4. law25_1920_a001  (0.840)
   مادة 1 تجب النفقة للزوجة على زوجها من تاريخ العقد الصحيح إذا سلمت نفسها إليه ولو...
5. law25_1920_a005  (0.840)
   مادة 5 إذا كان الزوج غائبا غيبة قريبة فان كان له مال ظاهر نفذ عليه الحم بالنفقة ...


In [11]:
def search(question: str, k: int) -> list[dict]:
    q_vec = model.encode(
        ["query: " + normalize_ar(question)],
        normalize_embeddings=True,
    )
    scores, positions = index.search(q_vec, k)
    return [
        {"chunk_id": chunks[pos]["id"], "score": round(float(score), 4)}
        for pos, score in zip(positions[0], scores[0])
    ]


results = search("هل يمكن حبس من يمتنع عن دفع النفقة؟", 5)
for r in results:
    print(r)

{'chunk_id': 'law1_2000_a076bis', 'score': 0.8514}
{'chunk_id': 'law25_1920_a004', 'score': 0.8461}
{'chunk_id': 'law1_2000_a079', 'score': 0.8423}
{'chunk_id': 'law25_1920_a001', 'score': 0.8403}
{'chunk_id': 'law25_1920_a005', 'score': 0.8401}


In [12]:
import random

random.seed(42)

hits = 0
misses = []

for c in chunks:
    lines = [l.strip() for l in c["text"].split("\n") if len(l.strip()) > 30]
    if not lines:
        continue
    sentence = random.choice(lines)

    results = search(sentence, 5)
    top_ids = [r["chunk_id"] for r in results]

    if c["id"] in top_ids:
        hits += 1
    else:
        misses.append((c["id"], sentence[:60], top_ids[0]))

tested = hits + len(misses)
print(f"نجح: {hits} من {tested}  ({hits / tested:.1%})")
print()
print("المواد اللي فشلت:")
for cid, sent, got in misses:
    print(f"  {cid}  ←  جاب {got} بداله")
    print(f"     الجملة: {sent}...")

نجح: 162 من 184  (88.0%)

المواد اللي فشلت:
  law25_1929_a006  ←  جاب law25_1929_a007 بداله
     الجملة: حكمين وقضى على الوجه المبين بالمواد 7،8،9،10،11....
  law25_1929_a008  ←  جاب law25_1929_a011 بداله
     الجملة: فان لم يقدما تقرير هما اعتبرها غير متفقين....
  law25_1929_a015  ←  جاب law25_1929_a017 بداله
     الجملة: والمتوفى عنها زوجها اذا آتت به لأكثر من سنة من وقت الطلاق او...
  law25_1929_a024  ←  جاب law1_2000_a005 بداله
     الجملة: ومسائل متعلقة بالأحوال الشخصية....
  law1_2000_a008  ←  جاب law25_1920_a010 بداله
     الجملة: ناظرا بصفة مؤقتة إلي أن يفصل فى الدعوى بحكم نهائي ....
  law1_2000_a009  ←  جاب decree1088_2000_a019 بداله
     الجملة: للحصول على اذن ، وسلب اى من هذه الحقوق او وقفها او الحد منها...
  law1_2000_a012  ←  جاب law1_2000_a040 بداله
     الجملة: بالمادة (41) من هذا القانون . وتتخذ النيابة العامة على وجه ا...
  law1_2000_a021  ←  جاب law25_1929_a011bis بداله
     الجملة: وتطبق جميع الأحكام السابقة فى حالة طلب الزوجة تطليق نفسها إذ...
  law1_2000_a023  ←  ج